# 2.1 - Perceptrón multicapa: XOR paso a paso con NumPy

En este notebook entrenaremos una red **2–2–1** para resolver la puerta XOR:

* **Dos entradas**, una por variable. La capa de entrada recibe los datos.
* **Dos neuronas ocultas**, que calculan sumas ponderadas y aplican una activación no lineal.
* **Una neurona de salida**, que estima la probabilidad de la clase 1.

El botón **«Ejecutar 1 epoch»** realizará una sola actualización de gradiente descendente utilizando los cuatro puntos. Observaremos las regiones de clasificación, las rectas de las neuronas ocultas, la pérdida y un **diagrama de la red con sus nueve pesos y sus cambios por epoch**.

La implementación del modelo y del entrenamiento utiliza **solo NumPy**. pandas, Matplotlib e ipywidgets se utilizan para las tablas, las gráficas y los controles, como en el notebook 1.4.

---
# 1. La puerta XOR y sus cuatro puntos

XOR devuelve 1 cuando las dos entradas son diferentes y 0 cuando son iguales.

| $x_1$ | $x_2$ | $y$ | Color de la clase real |
| --- | --- | --- | --- |
| 0 | 0 | 0 | Azul |
| 0 | 1 | 1 | Rojo |
| 1 | 0 | 1 | Rojo |
| 1 | 1 | 0 | Azul |

Una sola recta no separa las clases en estas coordenadas. Una red con dos neuronas ocultas puede combinar dos transformaciones no lineales para clasificar los cuatro puntos.

In [1]:
from io import BytesIO
import base64
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Circle, FancyArrowPatch
from matplotlib.ticker import MaxNLocator
import ipywidgets as widgets
from IPython.display import display

X = np.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y = np.array([[0.], [1.], [1.], [0.]])
N = len(X)

dataset = pd.DataFrame(X.astype(int), columns=['X₁', 'X₂'])
dataset['Clase real'] = y.ravel().astype(int)
display(dataset)

,X₁,X₂,Clase real
0,0,0,0
1,0,1,1
2,1,0,1
3,1,1,0


---
# 2. Arquitectura, activaciones y pesos

```text
                       N₁: tanh
                      /        \
Entradas: x₁, x₂                    S₁: sigmoide → p(clase 1)
                      \        /
                       N₂: tanh

Cada neurona oculta y la neurona de salida tienen su propio sesgo.
```

En este ejemplo usaremos **tanh en la capa oculta**, **sigmoide en la salida** y **entropía cruzada binaria**. Esta es la configuración de todo el notebook. Las sigmoides ocultas y la pérdida cuadrática del ejemplo numérico de la PPT corresponden a otro ejemplo.

Para cada neurona oculta $i$:

$$E_i^{(1)}=w_{i,0}^{(1)}\cdot1+w_{i,1}^{(1)}x_1+w_{i,2}^{(1)}x_2,
\qquad N_i=\tanh(E_i^{(1)})$$

La salida de la red es:

$$E_1^{(2)}=w_{1,0}^{(2)}\cdot1+w_{1,1}^{(2)}N_1+w_{1,2}^{(2)}N_2,
\qquad p=\sigma(E_1^{(2)})$$

Predecimos clase 1 si $p\geq0,5$ y clase 0 en otro caso.

**Las dos rectas de las neuronas ocultas.** Cada ecuación $E_i^{(1)}=0$ define un hiperplano, que aquí es una recta. A un lado, tanh produce valores positivos; al otro, negativos. La salida combina ambas respuestas. Con estas activaciones suaves, la frontera final $p=0,5$ puede ser curva y no tiene por qué coincidir con esas dos rectas.

**Notación de las matrices.** Incorporaremos el bias mediante una columna constante de valor 1. Como en el borrador de la PPT, las filas de cada matriz de pesos representan los orígenes y sus columnas los destinos:

$$A_0=[\mathbf{1},X],\quad W_1\in\mathbb{R}^{3\times2},\quad
H=\tanh(A_0W_1),\quad A_1=[\mathbf{1},H],\quad W_2\in\mathbb{R}^{3\times1}$$

Así, `W1[j, i]` almacena $w_{i+1,j}^{(1)}$. La primera fila de cada matriz contiene los **pesos del bias**, que sí son entrenables; la entrada constante 1 se mantiene fija. En total hay **9 parámetros**.

---
# 3. Inicialización aleatoria

El notebook comienza con **pesos aleatorios reproducibles**. Los pesos del bias empiezan en cero y también se ajustan al entrenar.

### Pesos distintos: aprender XOR

Inicializaremos los pesos de las conexiones con valores aleatorios reproducibles; los pesos de los bias pueden empezar en cero. Esto rompe la simetría entre las neuronas ocultas.

La configuración didáctica **semilla 5752**, desviación inicial **0,5** y **α = 3,0** está seleccionada para conseguir **4/4 aciertos por primera vez en la quinta época**, con una actualización real por época. No hay entrenamiento oculto ni pesos de una solución precargada.

La pérdida todavía será aproximadamente **0,5910** en esa época: acertar las cuatro clases con el umbral 0,5 no equivale a producir probabilidades cercanas a 0 y 1. Sigue avanzando para observar cómo cambia la confianza.

> Cinco épocas es el resultado de esta configuración concreta, no una garantía general. Cambiar la semilla, la tasa o las activaciones puede requerir más épocas o impedir que el modelo aprenda. La tasa 3,0 se ha elegido para este pequeño ejemplo; no es una recomendación general.

Para reproducir las capturas de la PPT, utiliza **semilla 42 y α = 3,0**: el primer 4/4 se obtiene en el epoch 27 y, tras 50 épocas, la pérdida es aproximadamente **0,0479**. La semilla predeterminada 5752 se conserva para el recorrido corto de cinco épocas.

---
# 4. Forward y función de pérdida

La entropía cruzada binaria usa las probabilidades de las cuatro observaciones:

$$L=-\frac{1}{N}\sum_{i=1}^{N}\left[y_i\log(p_i)+(1-y_i)\log(1-p_i)\right]$$

Para evitar calcular logaritmos de probabilidades muy cercanas a cero o uno, la implementamos de forma equivalente a partir de la suma de entrada de la salida $Z_2$:

$$L=\frac{1}{N}\sum_{i=1}^{N}\left[\log(1+e^{Z_{2,i}})-y_iZ_{2,i}\right]$$

`np.logaddexp(0, Z2)` calcula el primer término de forma numéricamente estable.

In [2]:
def sigmoide(z):
    """Sigmoide estable, tanto para números positivos como negativos."""
    z = np.asarray(z, dtype=float)
    p = np.empty_like(z)
    positivos = z >= 0
    p[positivos] = 1 / (1 + np.exp(-z[positivos]))
    exp_z = np.exp(z[~positivos])
    p[~positivos] = exp_z / (1 + exp_z)
    return p


def forward(X_datos, parametros):
    """Calcula todas las activaciones sin modificar ningún parámetro."""
    A0 = np.column_stack([np.ones(len(X_datos)), X_datos])
    Z1 = A0 @ parametros['W1']
    H = np.tanh(Z1)
    A1 = np.column_stack([np.ones(len(X_datos)), H])
    Z2 = A1 @ parametros['W2']
    p = sigmoide(Z2)
    return {'A0': A0, 'Z1': Z1, 'H': H, 'A1': A1, 'Z2': Z2, 'p': p}


def calcular_metricas(X_datos, y_real, parametros):
    """Evalúa los mismos parámetros que se mostrarán en la gráfica."""
    cache = forward(X_datos, parametros)
    perdida = np.mean(np.logaddexp(0, cache['Z2']) - y_real * cache['Z2'])
    clases = (cache['p'] >= 0.5).astype(int)
    aciertos = int(np.sum(clases == y_real))
    return float(perdida), aciertos, cache

---
# 5. Backpropagation y una actualización por epoch

Para sigmoide con entropía cruzada, el gradiente respecto de $Z_2$ se simplifica. En estas matrices incluimos ya el promedio de los $N=4$ ejemplos:

$$D_2=\frac{p-y}{N},\qquad G_2=A_1^\mathsf{T}D_2$$

Propagamos el gradiente hacia las neuronas ocultas. La derivada de tanh es $1-H^2$:

$$D_1=\left(D_2W_{2,\mathrm{sin\ bias}}^\mathsf{T}\right)\odot(1-H^2),
\qquad G_1=A_0^\mathsf{T}D_1$$

El símbolo $\odot$ representa multiplicación elemento a elemento. Todos los gradientes utilizan los **pesos anteriores**. Después los actualizamos simultáneamente:

$$W_1^{new}=W_1^{old}-\alpha G_1,\qquad W_2^{new}=W_2^{old}-\alpha G_2$$

Aquí trabajamos directamente con el gradiente y lo **restamos**. La señal de corrección con signo contrario usada en la PPT permite escribir una actualización equivalente sumando el incremento.

```text
Para cada epoch:
    Forward de los cuatro puntos con los pesos actuales
    Calcular la pérdida y los gradientes mediante backpropagation
    Actualizar simultáneamente los nueve parámetros una sola vez
    Evaluar la red actualizada y añadir un punto a la gráfica de pérdida
```

In [3]:
def calcular_gradientes(X_datos, y_real, parametros):
    """Backpropagation por lote: aún no actualiza ningún peso."""
    cache = forward(X_datos, parametros)
    D2 = (cache['p'] - y_real) / len(X_datos)
    G2 = cache['A1'].T @ D2

    # La fila 0 de W2 corresponde al bias: no es una neurona oculta.
    D1 = (D2 @ parametros['W2'][1:, :].T) * (1 - cache['H'] ** 2)
    G1 = cache['A0'].T @ D1
    return {'W1': G1, 'W2': G2}, cache


def un_epoch(X_datos, y_real, parametros, alpha):
    """Un recorrido de los cuatro datos y UNA actualización por matriz."""
    gradientes, cache_anterior = calcular_gradientes(X_datos, y_real, parametros)
    nuevos = {nombre: parametros[nombre] - alpha * gradientes[nombre]
              for nombre in parametros}
    return nuevos, gradientes, cache_anterior


SEMILLA_INICIAL = 5752
ALPHA_INICIAL = 3.0
ESCALA_INICIAL = 0.5


def inicializar_parametros(semilla=SEMILLA_INICIAL):
    """Crea conexiones aleatorias y sesgos a cero, sin entrenar ni utilizar y."""
    generador = np.random.default_rng(semilla)
    W1 = np.zeros((3, 2), dtype=float)
    W2 = np.zeros((3, 1), dtype=float)
    W1[1:, :] = generador.normal(0, ESCALA_INICIAL, size=(2, 2))
    W2[1:, :] = generador.normal(0, ESCALA_INICIAL, size=(2, 1))
    return {'W1': W1, 'W2': W2}

In [4]:
def crear_figura_red(parametros, anteriores, epoch, indice_patron=1, inicial=False):
    """Dibuja los nueve pesos y un forward de la observación seleccionada."""
    entrada = X[indice_patron:indice_patron + 1]
    cache = forward(entrada, parametros)
    probabilidad = float(cache['p'][0, 0])
    clase = int(probabilidad >= 0.5)
    posiciones = {
        'I0': (1.0, 4.65), 'I1': (1.0, 3.15), 'I2': (1.0, 1.0),
        'N0': (7.5, 4.65), 'N1': (6.2, 3.15), 'N2': (6.2, 1.0),
        'S1': (12.1, 2.2),
    }
    # Matriz, fila de origen, columna de destino, nodos y posición de etiqueta.
    conexiones = [
        ('W1', 0, 0, 'I0', 'N1', (4.0, 4.0)),
        ('W1', 0, 1, 'I0', 'N2', (2.1, 4.0)),
        ('W1', 1, 0, 'I1', 'N1', (3.6, 3.25)),
        ('W1', 1, 1, 'I1', 'N2', (2.9, 2.05)),
        ('W1', 2, 0, 'I2', 'N1', (4.6, 2.05)),
        ('W1', 2, 1, 'I2', 'N2', (3.5, 0.78)),
        ('W2', 0, 0, 'N0', 'S1', (10.05, 3.75)),
        ('W2', 1, 0, 'N1', 'S1', (8.5, 2.85)),
        ('W2', 2, 0, 'N2', 'S1', (9.05, 1.18)),
    ]
    fig, ax = plt.subplots(figsize=(13.6, 5.7))
    ax.set(xlim=(0, 13.3), ylim=(-0.6, 5.9))
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(f'Red 2–2–1 · epoch {epoch} · entrada '
                 f'({int(entrada[0, 0])}, {int(entrada[0, 1])}), '
                 f'y = {int(y[indice_patron, 0])}', fontsize=15, pad=14)
    for texto, x in [('Capa de entrada', 1.0), ('Capa oculta · tanh', 6.2),
                      ('Salida · sigmoide', 12.0)]:
        ax.text(x, 5.48, texto, ha='center', fontsize=12, fontweight='bold', color='#1f2a44')

    for nombre, origen, destino, nodo_origen, nodo_destino, posicion_texto in conexiones:
        valor = float(parametros[nombre][origen, destino])
        previo = float(anteriores[nombre][origen, destino])
        incremento = valor - previo
        color = '#64748b' if inicial or incremento == 0 else '#15803d' if incremento > 0 else '#c2410c'
        a = np.array(posiciones[nodo_origen])
        b = np.array(posiciones[nodo_destino])
        direccion = (b - a) / np.linalg.norm(b - a)
        inicio = a + direccion * 0.4
        final = b - direccion * (0.49 if nodo_destino == 'S1' else 0.4)
        ax.add_patch(FancyArrowPatch(inicio, final, arrowstyle='-|>', mutation_scale=13,
                                     linewidth=1.5, color=color, alpha=0.85, zorder=1))
        capa = 1 if nombre == 'W1' else 2
        etiqueta = rf'$w_{{{destino + 1},{origen}}}^{{({capa})}}={valor:.4f}$'
        subtitulo = 'Inicial' if inicial else rf'$\Delta w={incremento:+.4f}$'
        tx, ty = posicion_texto
        # Cajas ajustadas al texto; los valores no quedan ocultos por las flechas.
        ax.text(tx, ty + 0.09, etiqueta, ha='center', va='center', fontsize=10.5,
                color='#1f2a44', bbox={'facecolor': 'white', 'edgecolor': 'none', 'pad': 1.2}, zorder=4)
        ax.text(tx, ty - 0.2, subtitulo, ha='center', va='center', fontsize=9,
                color=color, bbox={'facecolor': 'white', 'edgecolor': 'none', 'pad': 1}, zorder=4)

    nodos = {
        'I0': ('$I_0$\n1', '#eef0f4', '#64748b'),
        'I1': (f'$I_1=x_1$\n{entrada[0, 0]:.0f}', '#e6eeff', '#2563eb'),
        'I2': (f'$I_2=x_2$\n{entrada[0, 1]:.0f}', '#e6eeff', '#2563eb'),
        'N0': ('$N_0$\n1', '#eef0f4', '#64748b'),
        'N1': (f'$N_1$\n{cache["H"][0, 0]:.4f}', '#ddf5e8', '#16a34a'),
        'N2': (f'$N_2$\n{cache["H"][0, 1]:.4f}', '#ebe4ff', '#7c3aed'),
        'S1': (f'$S_1$\np={probabilidad:.4f}\nŷ={clase}', '#f8fafc', '#1f2a44'),
    }
    for nombre, (texto, relleno, borde) in nodos.items():
        x_nodo, y_nodo = posiciones[nombre]
        radio = 0.47 if nombre == 'S1' else 0.38
        ax.add_patch(Circle((x_nodo, y_nodo), radio, facecolor=relleno,
                             edgecolor=borde, linewidth=1.7, zorder=5))
        ax.text(x_nodo, y_nodo, texto, ha='center', va='center', fontsize=10.5,
                color='#1f2a44', zorder=6)
    leyenda = [Line2D([], [], color='#15803d', label='El peso aumenta'),
                Line2D([], [], color='#c2410c', label='El peso disminuye'),
                Line2D([], [], color='#64748b', label='Inicial / sin cambios')]
    ax.legend(handles=leyenda, loc='lower center', bbox_to_anchor=(0.5, -0.02),
               ncol=3, fontsize=10, frameon=False)
    ax.text(0.15, -0.13, 'I₀ = N₀ = 1: entradas constantes del bias. '
                         'Δw compara con la época anterior.', fontsize=9.5, color='#64748b')
    fig.tight_layout()
    return fig


def figura_como_html(fig, descripcion):
    """Convierte una figura en una imagen visible sin depender de un backend interactivo."""
    with BytesIO() as imagen:
        fig.savefig(imagen, format='png', dpi=125, bbox_inches='tight')
        datos_imagen = base64.b64encode(imagen.getvalue()).decode('ascii')
    plt.close(fig)
    return (f'<img src="data:image/png;base64,{datos_imagen}" alt="{descripcion}" '
            'style="width:100%;height:auto;display:block;">')


---
# 6. Entrenamiento interactivo

1. Ejecuta las celdas en orden. El simulador empieza con **pesos aleatorios**, en el epoch 0, antes de entrenar.
2. Pulsa **«Ejecutar 1 epoch»**. El diagrama de la red muestra los pesos actuales y su cambio **Δw** respecto a la época anterior: **verde si aumentan**, **naranja si disminuyen** y gris si no cambian.
3. Con α = 3,0 y semilla 5752, cinco pulsaciones consiguen 4/4 aciertos. El dibujo, las regiones de clasificación y la pérdida utilizan siempre los mismos parámetros actualizados.
4. En **«Entrada del diagrama»** elige uno de los cuatro patrones XOR para ver los valores de las entradas, las activaciones ocultas y la predicción. Esta selección no entrena ni cambia los pesos: estos son comunes a las cuatro observaciones.
5. **Reiniciar** restaura los parámetros aleatorios de la semilla seleccionada. Cambiar la semilla también reinicia el entrenamiento; α se conserva.

**Cómo leer la gráfica:** los cuatro puntos conservan el color de su clase real: azul para 0 y rojo para 1. El fondo representa la clase predicha por la red; una cruz negra señala un punto mal clasificado. La anotación muestra la clase predicha y su probabilidad de clase 1. Las rectas verde y violeta son $E_1^{(1)}=0$ y $E_2^{(1)}=0$. La línea negra, cuando existe en la zona mostrada, es la frontera final $p=0,5$.

Abre **«Pesos y última actualización»** para ver los nueve pesos anteriores, sus gradientes y los nuevos. La tabla resalta los valores que cambian; en el diagrama, el color de Δw distingue aumentos y disminuciones. Las cifras se presentan con cuatro decimales; el cálculo mantiene la precisión completa.

> La pérdida y los aciertos se calculan sobre los cuatro patrones XOR usados para entrenar. Estamos estudiando el funcionamiento de la red en una tabla de verdad completa, no estimando su generalización en un problema real.

In [5]:
estado = {}

boton_epoch = widgets.Button(description='Ejecutar 1 epoch', button_style='primary',
                             icon='step-forward', layout=widgets.Layout(width='180px'))
boton_reiniciar = widgets.Button(description='Reiniciar', icon='refresh',
                                 layout=widgets.Layout(width='125px'))
alpha = widgets.FloatText(value=ALPHA_INICIAL, description='α:', step=0.1,
                          style={'description_width': '25px'},
                          layout=widgets.Layout(width='150px'))
semilla = widgets.BoundedIntText(value=SEMILLA_INICIAL, min=0, max=2**32 - 1,
                                 description='Semilla:',
                                 style={'description_width': '65px'},
                                 layout=widgets.Layout(width='190px'))
patron = widgets.Dropdown(
    options=[(f'({int(fila[0])}, {int(fila[1])}) → clase real {int(y[i, 0])}', i)
             for i, fila in enumerate(X)], value=1, description='Entrada del diagrama:',
    style={'description_width': '145px'}, layout=widgets.Layout(width='420px'))
salida_red = widgets.HTML(layout=widgets.Layout(width='100%'))
resumen = widgets.HTML()
aviso = widgets.HTML()
salida_grafica = widgets.HTML(layout=widgets.Layout(width='100%'))
tabla_predicciones = widgets.HTML()
tabla_pesos = widgets.HTML()
detalle = widgets.Accordion(children=[tabla_pesos], selected_index=None)
detalle.set_title(0, 'Pesos y última actualización')

xx, yy = np.meshgrid(np.linspace(-0.45, 1.45, 180), np.linspace(-0.45, 1.45, 180))
malla = np.column_stack([xx.ravel(), yy.ravel()])


def registrar_metricas():
    perdida, aciertos, _ = calcular_metricas(X, y, estado['parametros'])
    estado['epochs'].append(estado['epoch'])
    estado['perdidas'].append(perdida)
    estado['aciertos'].append(aciertos)
    if aciertos == N and estado['primer_epoch_correcto'] is None:
        estado['primer_epoch_correcto'] = estado['epoch']


def html_tabla_predicciones(cache):
    probabilidades = cache['p'].ravel()
    clases = (probabilidades >= 0.5).astype(int)
    filas = []
    for i in range(N):
        correcta = clases[i] == int(y[i, 0])
        color = '#2563eb' if y[i, 0] == 0 else '#dc2626'
        filas.append(f"<tr><td>{int(X[i, 0])}</td><td>{int(X[i, 1])}</td>"
                     f"<td style='color:{color};font-weight:bold'>{int(y[i, 0])}</td>"
                     f"<td>{probabilidades[i]:.4f}</td><td>{clases[i]}</td>"
                     f"<td>{'Sí' if correcta else 'No'}</td></tr>")
    return ("<b>Predicciones con los pesos actuales</b><table class='xor-tabla'>"
            "<tr><th>x₁</th><th>x₂</th><th>Clase real</th><th>p(clase 1)</th>"
            "<th>Clase predicha</th><th>Acierto</th></tr>" + ''.join(filas) + '</table>')


def html_tabla_pesos():
    filas = []
    for nombre, capa in [('W1', 1), ('W2', 2)]:
        actuales = estado['parametros'][nombre]
        anteriores = estado['anteriores'][nombre]
        gradientes = estado['gradientes']
        for origen in range(actuales.shape[0]):
            for destino in range(actuales.shape[1]):
                viejo = anteriores[origen, destino]
                nuevo = actuales[origen, destino]
                cambio = nuevo != viejo
                color = '#0a7a3b' if cambio else '#1f2a44'
                gradiente = '—' if gradientes is None else f"{gradientes[nombre][origen, destino]:.4f}"
                etiqueta = f'w<sub>{destino + 1},{origen}</sub><sup>({capa})</sup>'
                filas.append(f'<tr><td>{etiqueta}</td><td>{viejo:.4f}</td>'
                             f'<td>{gradiente}</td><td style="color:{color};'
                             f'font-weight:{"bold" if cambio else "normal"}">{nuevo:.4f}</td></tr>')
    return ("<table class='xor-tabla'><tr><th>Peso</th><th>Anterior</th>"
            "<th>Gradiente ∂L/∂w</th><th>Actual</th></tr>" + ''.join(filas) +
            '</table><small>Fila de origen 0: peso del bias. '
            'Los gradientes se calculan antes de actualizar cualquier capa.</small>')


def crear_figura():
    parametros = estado['parametros']
    cache_malla = forward(malla, parametros)
    z_malla = cache_malla['Z2'].reshape(xx.shape)
    regiones = (cache_malla['p'] >= 0.5).reshape(xx.shape).astype(int)
    p_puntos = forward(X, parametros)['p'].ravel()
    clases = (p_puntos >= 0.5).astype(int)
    colores = ['#2563eb', '#dc2626']
    fig, (ax_clases, ax_perdida) = plt.subplots(1, 2, figsize=(12.8, 5.5))
    ax_clases.pcolormesh(xx, yy, regiones, cmap=ListedColormap(['#dbeafe', '#fee2e2']),
                         vmin=0, vmax=1, shading='auto', rasterized=True)
    leyenda = []
    for clase in [0, 1]:
        puntos = y.ravel() == clase
        ax_clases.scatter(X[puntos, 0], X[puntos, 1], c=colores[clase],
                          edgecolors='black', s=150, linewidths=0.8, zorder=4)
        leyenda.append(Line2D([], [], color=colores[clase], marker='o', linestyle='',
                              label=f'Clase real {clase}', markersize=7))
    for j, color in enumerate(['#16a34a', '#7c3aed']):
        valores = cache_malla['Z1'][:, j].reshape(xx.shape)
        if valores.min() < 0 < valores.max():
            ax_clases.contour(xx, yy, valores, levels=[0], colors=color,
                              linestyles='--', linewidths=1.4)
            leyenda.append(Line2D([], [], color=color, linestyle='--',
                                  label=f'Neurona oculta {j + 1}: E = 0'))
    if z_malla.min() < 0 < z_malla.max():
        ax_clases.contour(xx, yy, z_malla, levels=[0], colors='#1f2a44', linewidths=2)
        leyenda.append(Line2D([], [], color='#1f2a44', label='Salida: p = 0,5'))
    errores = clases != y.ravel().astype(int)
    if errores.any():
        ax_clases.scatter(X[errores, 0], X[errores, 1], marker='x', c='black',
                          s=75, linewidths=2, zorder=5)
        leyenda.append(Line2D([], [], color='black', marker='x', linestyle='',
                              label='Clasificación incorrecta'))
    for i, (x1, x2) in enumerate(X):
        ax_clases.annotate(f'ŷ={clases[i]}\np={p_puntos[i]:.4f}', (x1, x2),
                           xytext=(9, 9), textcoords='offset points', fontsize=8,
                           bbox={'facecolor': 'white', 'alpha': 0.8, 'edgecolor': 'none', 'pad': 2},
                           zorder=6)
    ax_clases.set(xlabel='x₁', ylabel='x₂', xlim=(-0.45, 1.45), ylim=(-0.45, 1.45),
                  title=f'XOR · red actual, epoch {estado["epoch"]}')
    ax_clases.set_aspect('equal')
    ax_clases.legend(handles=leyenda, loc='lower center', bbox_to_anchor=(0.5, -0.37),
                      fontsize=8, ncol=2)
    ax_perdida.plot(estado['epochs'], estado['perdidas'], 'o-', color='darkorange',
                    markersize=5)
    ax_perdida.set(xlabel='Epoch', ylabel='Entropía cruzada binaria', title='Pérdida por epoch',
                   ylim=(0, max(0.8, max(estado['perdidas']) * 1.1)))
    ax_perdida.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax_perdida.grid(alpha=0.3)
    fig.tight_layout()
    return fig


def actualizar_vista():
    perdida, aciertos, cache = calcular_metricas(X, y, estado['parametros'])
    primero = estado['primer_epoch_correcto']
    texto_primero = 'Todavía no' if primero is None else str(primero)
    resumen.value = (f"<div style='font-size:19px;line-height:1.8'><b>Epoch:</b> {estado['epoch']} "
                     f"&nbsp; <b>Pérdida:</b> {perdida:.4f} &nbsp; <b>Aciertos:</b> {aciertos}/4 "
                     f"&nbsp; <b>α:</b> {alpha.value:g}<br>"
                     f"<b>Primer epoch con 4/4 aciertos:</b> {texto_primero}</div>")
    aviso.value = ("<div style='background:#e8f0fe;padding:10px;color:#1f2a44'>"
                   '<b>Inicialización aleatoria.</b> Una pulsación equivale a una época '
                   'y una actualización real. Cambia la semilla para comparar '
                   'distintos puntos de partida.</div>')
    tabla_predicciones.value = html_tabla_predicciones(cache)
    tabla_pesos.value = html_tabla_pesos()
    fig_red = crear_figura_red(estado['parametros'], estado['anteriores'], estado['epoch'],
                               patron.value, inicial=estado['gradientes'] is None)
    salida_red.value = figura_como_html(fig_red, 'Red neuronal 2–2–1 con pesos actuales y cambios por epoch')
    fig = crear_figura()
    with BytesIO() as imagen:
        fig.savefig(imagen, format='png', dpi=120, bbox_inches='tight')
        imagen_base64 = base64.b64encode(imagen.getvalue()).decode('ascii')
    plt.close(fig)
    salida_grafica.value = (f'<img src="data:image/png;base64,{imagen_base64}" '
                            'alt="Regiones de XOR, rectas ocultas y pérdida por epoch" '
                            'style="width:100%;height:auto;display:block;">')


def ejecutar_epoch(_=None):
    if not np.isfinite(alpha.value) or alpha.value <= 0:
        aviso.value = '<b style="color:#b91c1c">Introduce una tasa α positiva y finita.</b>'
        return
    boton_epoch.disabled = True
    try:
        anteriores = {nombre: matriz.copy() for nombre, matriz in estado['parametros'].items()}
        with np.errstate(over='raise', invalid='raise'):
            nuevos, gradientes, _ = un_epoch(X, y, anteriores, alpha.value)
            if not all(np.isfinite(matriz).all() for matriz in nuevos.values()):
                raise FloatingPointError('Parámetros no finitos.')
            perdida_nueva, _, _ = calcular_metricas(X, y, nuevos)
            if not np.isfinite(perdida_nueva):
                raise FloatingPointError('Pérdida no finita.')
        estado.update(parametros=nuevos, anteriores=anteriores, gradientes=gradientes)
        estado['epoch'] += 1
        registrar_metricas()
        actualizar_vista()
    except FloatingPointError:
        aviso.value = ('<b style="color:#b91c1c">La tasa provoca valores no finitos. '
                       'Reduce α; no se ha aplicado este paso.</b>')
    finally:
        boton_epoch.disabled = False


def reiniciar(_=None):
    parametros = inicializar_parametros(semilla.value)
    estado.clear()
    estado.update(epoch=0, parametros=parametros,
                  anteriores={nombre: matriz.copy() for nombre, matriz in parametros.items()},
                  gradientes=None, epochs=[], perdidas=[], aciertos=[], primer_epoch_correcto=None)
    registrar_metricas()
    actualizar_vista()


boton_epoch.on_click(ejecutar_epoch)
boton_reiniciar.on_click(reiniciar)
semilla.observe(reiniciar, names='value')
alpha.observe(lambda cambio: actualizar_vista(), names='value')
patron.observe(lambda cambio: actualizar_vista(), names='value')

estilo = widgets.HTML("""<style>
.xor-tabla {border-collapse:collapse;margin:10px 0;font-size:15px;}
.xor-tabla th,.xor-tabla td {padding:7px 14px;border-bottom:1px solid #dadce0;text-align:right;}
.xor-tabla th {background:#e8f0fe;color:#1f2a44;}
</style>""")
controles = widgets.VBox([widgets.HBox([boton_epoch, boton_reiniciar, alpha, semilla]), patron])
display(estilo, controles, resumen, aviso, salida_grafica, salida_red, tabla_predicciones, detalle)
reiniciar()

HTML(value='<style>\n.xor-tabla {border-collapse:collapse;margin:10px 0;font-size:15px;}\n.xor-tabla th,.xor-t…

HTML(value='')

HTML(value='')

HTML(value='', layout=Layout(width='100%'))

HTML(value='', layout=Layout(width='100%'))

HTML(value='')

Accordion(children=(HTML(value=''),), titles=('Pesos y última actualización',))